In [ ]:
# conda install -c conda-forge pandas numpy matplotlib plotly pytorch seaborn darts sktime scikit-learn -y
# pip install datasets transformers chronos-forecasting


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import random
from tqdm import tqdm
import polars as pl

# from autogluon.timeseries import TimeSeriesDataFrame
# from autogluon.timeseries import TimeSeriesPredictor

# Merge DF

In [ ]:
rain_df = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/Train/HII_station_2015-2020.csv")
gsmap_df = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/Train/GSMap_now_station_2015-2020.csv")
humid_df = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/Train/Humidity_2015-2020.csv")
pressure_df = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/Train/Pressure_2015-2020.csv")
temp_df = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/Train/Temperature_2015-2020.csv")

gsmap_df_test = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/Test/GSMap_now_station_2021-2024.csv")
humid_df_test = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/Test/Humidity_2021-2024.csv")
pressure_df_test = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/Test/Pressure_2021-2024.csv")
temp_df_test = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/Test/Temperature_2021-2024.csv")
rain_df_test = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/CleanedSolto8th.csv")

coordinate_df = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/Coor_HII_495sta.csv")

In [ ]:
print(rain_df.head())
print(gsmap_df.head())
print(humid_df.head())
print(pressure_df.head())
print(temp_df.head())

print(coordinate_df.head())

In [ ]:
def format_gs_map_dataset(df_humid, df_pressure, df_temperature, df_gsmap, df_rain=None):
    """
    Reshapes and merges humidity, pressure, temperature, and GSMap DataFrames
    into a single, easy-to-read format, with an optimized memory footprint.
    """

    id_vars = ['Year', 'Month', 'Day', 'Hour']

    # Helper function to get station columns
    def get_station_cols(df, id_cols):
        return [col for col in df.columns if col not in id_cols and col != 'Index']

    print("Melting Humidity Data...")
    humid_station_cols = get_station_cols(df_humid, id_vars)
    merged_df = df_humid.melt(
        id_vars=id_vars,
        value_vars=humid_station_cols,
        var_name='Station',
        value_name='Humidity'
    )
    # Hint to Python to free memory if df_humid is no longer needed
    del df_humid

    print("Melting and Merging Pressure Data...")
    pressure_station_cols = get_station_cols(df_pressure, id_vars)
    df_pressure_melted = df_pressure.melt(
        id_vars=id_vars,
        value_vars=pressure_station_cols,
        var_name='Station',
        value_name='Pressure'
    )
    del df_pressure # Free memory from original df_pressure
    merged_df = pd.merge(merged_df, df_pressure_melted, on=id_vars + ['Station'], how='left')
    del df_pressure_melted # Free memory from the melted pressure df

    print("Melting and Merging Temperature Data...")
    temperature_station_cols = get_station_cols(df_temperature, id_vars)
    df_temperature_melted = df_temperature.melt(
        id_vars=id_vars,
        value_vars=temperature_station_cols,
        var_name='Station',
        value_name='Temperature'
    )
    del df_temperature
    merged_df = pd.merge(merged_df, df_temperature_melted, on=id_vars + ['Station'], how='left')
    del df_temperature_melted

    print("Melting and Merging GSMap Data...")
    gsmap_station_cols = get_station_cols(df_gsmap, id_vars)
    df_gsmap_melted = df_gsmap.melt(
        id_vars=id_vars,
        value_vars=gsmap_station_cols,
        var_name='Station',
        value_name='GSMap'
    )
    del df_gsmap
    merged_df = pd.merge(merged_df, df_gsmap_melted, on=id_vars + ['Station'], how='left')
    del df_gsmap_melted

    if df_rain is not None:
        print("Melting and Merging Rain Data...")
        rain_station_cols = get_station_cols(df_rain, id_vars)
        df_rain_melted = df_rain.melt(
            id_vars=id_vars,
            value_vars=rain_station_cols,
            var_name='Station',
            value_name='HII'
        )
        del df_rain
        merged_df = pd.merge(merged_df, df_rain_melted, on=id_vars + ['Station'], how='left')
        del df_rain_melted

        # Add an 'index' column as requested, using the default pandas index
        merged_df = merged_df.reset_index()

        # Reorder columns to match the desired output
        final_columns = [
            'index', 'Year', 'Month', 'Day', 'Hour', 'Station',
            'Temperature', 'Pressure', 'Humidity', 'GSMap', 'HII'
        ]

    else:
        merged_df = merged_df.reset_index()

        # Reorder columns to match the desired output
        final_columns = [
            'index', 'Year', 'Month', 'Day', 'Hour', 'Station',
            'Temperature', 'Pressure', 'Humidity', 'GSMap'
        ]


    merged_df = merged_df[final_columns]
    merged_df = merged_df.drop(columns = ['index'])

    # merged_df['timestamp'] = merged_df['Year'].astype(str) + '-' + merged_df['Month'].astype(str).str.zfill(2) + '-' + merged_df['Day'].astype(str).str.zfill(2) + ' ' + merged_df['Hour'].astype(str).str.zfill(2) + ':00:00'
    # merged_df['timestamp'] = pd.to_datetime(merged_df['timestamp'])
    # merged_df = merged_df.drop(columns=["Year", "Month", "Day", "Hour"])
    
    print("Data formatting complete!")
    return merged_df

In [ ]:
def insert_location(df, df_location):
    
    df_location = df_location.set_index('Station')    
    # Merge Dataframe with latitude and longtitude
    df = pd.merge(df, df_location, left_on="Station", right_index=True, how="left")
    # Drop Index
    
    return df

In [ ]:
df_train = format_gs_map_dataset(
    df_humid=humid_df,
    df_pressure=pressure_df,
    df_temperature=temp_df,
    df_gsmap=gsmap_df,
    df_rain=rain_df
)

# df_train = insert_location(df_train, coordinate_df)

# df_train = TimeSeriesDataFrame.from_data_frame(df_train,
#                                                id_column='Station', 
#                                                timestamp_column='timestamp',
#                                                # static_features_df=static_feature_df,
#                                                )

In [ ]:
test_df = format_gs_map_dataset(
    df_humid=humid_df_test,
    df_pressure=pressure_df_test,
    df_temperature=temp_df_test,
    df_gsmap=gsmap_df_test,
    df_rain=rain_df_test
)

In [ ]:
test_df = test_df.dropna()
test_df = test_df.reset_index(drop=True)
test_df.isnull().sum()

In [ ]:
test_df

In [ ]:
train_df = df_train.copy()
df_train = df_train.drop(columns = ['HII'])

In [ ]:
df_train

# EDA

In [ ]:
station_list = df_train['Station'].unique().tolist()

random_station = random.choice(station_list)
print(f"Randomly selected station: {random_station}")

### visualize

#### visualize by hour

In [ ]:
df_station = df_train[df_train['Station'] == random_station].copy()

df_station['Datetime'] = pd.to_datetime(df_station[['Year', 'Month', 'Day', 'Hour']])

fig = make_subplots(rows=1, cols=5, shared_xaxes=True, subplot_titles=[
    "Temperature", "Pressure", "Humidity", "GSMap", "Rain"
])

fig = make_subplots(
    rows=5, cols=1,
    shared_xaxes=True,
    subplot_titles=["Temperature", "Pressure", "Humidity", "GSMap", "Rain"]
)

# Plot each variable
variables = ['Temperature', 'Pressure', 'Humidity', 'GSMap', 'Rain']
for i, var in enumerate(variables):
    fig.add_trace(
        go.Scatter(x=df_station['Datetime'], y=df_station[var], name=var, mode='lines'),
        row=i + 1, col=1
    )

# Update layout
fig.update_layout(
    height=1500, width=1000,
    title_text=f"Time Series for {random_station}",
    showlegend=False
)

fig.show()

#### visualize by date

In [ ]:
df_station = df_train[df_train['Station'] == random_station].copy()

df_station = df_station.groupby(['Year', 'Month', 'Day']).agg({
    'Temperature': 'mean',
    'Pressure': 'mean',
    'Humidity': 'mean',
    'GSMap': 'sum',
    'Rain': 'sum'
}).reset_index()

# Optional: combine into datetime
df_station['Datetime'] = pd.to_datetime(df_station[['Year', 'Month', 'Day']])

fig = make_subplots(rows=1, cols=5, shared_xaxes=True, subplot_titles=[
    "Temperature", "Pressure", "Humidity", "GSMap", "Rain"
])

fig = make_subplots(
    rows=5, cols=1,
    shared_xaxes=True,
    subplot_titles=["Temperature", "Pressure", "Humidity", "GSMap", "Rain"]
)

# Plot each variable
variables = ['Temperature', 'Pressure', 'Humidity', 'GSMap', 'Rain']
for i, var in enumerate(variables):
    fig.add_trace(
        go.Scatter(x=df_station['Datetime'], y=df_station[var], name=var, mode='lines'),
        row=i + 1, col=1
    )

# Update layout
fig.update_layout(
    height=1500, width=1000,
    title_text=f"Time Series for {random_station}",
    showlegend=False
)

fig.show()

### Correlation

In [ ]:
df_station = df_train[df_train['Station'] == random_station].copy()

df_station['Datetime'] = pd.to_datetime(df_station[['Year', 'Month', 'Day', 'Hour']])

correlation_matrix = df_station[['Temperature', 'Pressure', 'Humidity', 'GSMap', 'Rain']].corr()
print(correlation_matrix)

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Feature Correlation Heatmap")
plt.show()

In [ ]:
df_station = df_train[df_train['Station'] == random_station].copy()

df_station = df_station.groupby(['Year', 'Month', 'Day']).agg({
    'Temperature': 'mean',
    'Pressure': 'mean',
    'Humidity': 'mean',
    'GSMap': 'sum',
    'Rain': 'sum'
}).reset_index()

# Optional: combine into datetime
df_station['Datetime'] = pd.to_datetime(df_station[['Year', 'Month', 'Day']])

correlation_matrix = df_station[['Temperature', 'Pressure', 'Humidity', 'GSMap', 'Rain']].corr()
print(correlation_matrix)

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Feature Correlation Heatmap")
plt.show()

# Preprocess

In [ ]:
combined_df = pd.concat([df_train, test_df], ignore_index=True)

In [ ]:
combined_df = pl.from_pandas(combined_df)

In [ ]:
def create_rolling_features(
    df: pl.DataFrame,
    features: list,
    lag_days: list = [1, 3, 7, 30, 90],
    lag_hours: list = [3, 6, 12],
    group_by_col: str = 'Station'
) -> pl.DataFrame:
    
    if "Datetime" not in df.columns:
        df = df.with_columns(pl.datetime("Year", "Month", "Day", "Hour").alias("Datetime"))
    
    df = df.sort([group_by_col, "Datetime"])
    
    # Build all expressions at once
    rolling_exprs = []
    
    for feature in features:
        # For daily lags - convert days to hours (assuming hourly data)
        for lag in lag_days:
            lag_hours_equiv = lag * 24  # Convert days to hours
            rolling_exprs.extend([
                # Rolling mean over lag period - set min_periods to window size for nulls
                pl.col(feature).rolling_mean(lag_hours_equiv, min_periods=lag_hours_equiv).over(group_by_col)
                  .alias(f"{feature}_rolling_{lag}d_mean"),
                pl.col(feature).rolling_min(lag_hours_equiv, min_periods=lag_hours_equiv).over(group_by_col)
                  .alias(f"{feature}_rolling_{lag}d_min"),
                pl.col(feature).rolling_max(lag_hours_equiv, min_periods=lag_hours_equiv).over(group_by_col)
                  .alias(f"{feature}_rolling_{lag}d_max"),
                pl.col(feature).rolling_median(lag_hours_equiv, min_periods=lag_hours_equiv).over(group_by_col)
                  .alias(f"{feature}_rolling_{lag}d_median"),
                pl.col(feature).rolling_std(lag_hours_equiv, min_periods=lag_hours_equiv).over(group_by_col)
                  .alias(f"{feature}_rolling_{lag}d_std"),
            ])
        
        # For hourly lags - set min_periods to window size for nulls
        for lag in lag_hours:
            rolling_exprs.extend([
                pl.col(feature).rolling_mean(lag, min_periods=lag).over(group_by_col)
                  .alias(f"{feature}_rolling_{lag}h_mean"),
                pl.col(feature).rolling_std(lag, min_periods=lag).over(group_by_col)
                  .alias(f"{feature}_rolling_{lag}h_std"),
                pl.col(feature).rolling_median(lag, min_periods=lag).over(group_by_col)
                  .alias(f"{feature}_rolling_{lag}h_median"),
            ])
    
    return df.with_columns(rolling_exprs)

In [ ]:
def create_lagged_features(
    df: pl.DataFrame,
    features: list,
    lag_hours: list = [1, 2, 3, 4],
    group_by_col: str = 'Station'
) -> pl.DataFrame:
    
    # Create datetime column if it doesn't exist
    if "Datetime" not in df.columns:
        df = df.with_columns([
            pl.datetime(
                df["Year"], df["Month"], df["Day"], df["Hour"]
            ).alias("Datetime")
        ])
    
    df = df.sort([group_by_col, "Datetime"])

    exprs = [pl.col("*")]  # Keep original columns

    pbar = tqdm(total=len(features) * len(lag_hours), desc="Lag features (Polars)")

    for feature in features:
        for lag in lag_hours:
            exprs.append(
                pl.col(feature)
                  .shift(lag)
                  .over(group_by_col)
                  .alias(f"{feature}_lag_{lag}h")
            )
            pbar.update(1)

    pbar.close()
    return df.select(exprs)

In [ ]:
features = ['Temperature', 'Humidity', 'GSMap', 'Pressure']

combined_df = create_lagged_features(combined_df, features, group_by_col="Station")

In [ ]:
combined_df.columns

In [ ]:
combined_df.head

In [ ]:
combined_df = create_rolling_features(combined_df, features, group_by_col="Station")

In [ ]:
print(combined_df)

In [ ]:
combined_df.columns

In [ ]:
combined_df = combined_df.to_pandas()

In [ ]:
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)     # Show all rows (optional)
pd.set_option('display.width', 0)           # Auto line wrapping

print(combined_df.isnull().sum())

In [ ]:
pd.reset_option("all")

In [ ]:
combined_df = combined_df.drop(columns=['Datetime'])
combined_df = combined_df.dropna()
combined_df = combined_df.reset_index(drop=True)
combined_df.isnull().sum()

In [ ]:
combined_df

In [ ]:
combined_df.columns

### Merge back

In [ ]:
train_df

In [ ]:
train_df = pd.merge(
    combined_df,
    train_df[["Year", "Month", "Day", "Hour", "Station", "HII"]],
    on=["Year", "Month", "Day", "Hour", "Station"],
    how="inner"  # Only keep rows present in both DataFrames
)

In [ ]:
train_df

In [ ]:
test_df = pd.merge(
    combined_df,
    test_df[["Year", "Month", "Day", "Hour", "Station"]],
    on=["Year", "Month", "Day", "Hour", "Station"],
    how="inner"  # Only keep rows present in both DataFrames
)

In [ ]:
test_df

In [ ]:
eval_df = pd.merge(
    combined_df,
    test_df[["Year", "Month", "Day", "Hour", "Station", "HII"]],
    on=["Year", "Month", "Day", "Hour", "Station"],
    how="inner"  # Only keep rows present in both DataFrames
)

In [ ]:
eval_df

In [ ]:
# train_df.to_parquet('/project/ai901504-ai0004/501641_Big/week6/train_df_feature.parquet')
eval_df.to_parquet('/project/ai901504-ai0004/501641_Big/week6/eval_df_feature.parquet')
# test_df.to_parquet('/project/ai901504-ai0004/501641_Big/week6/test_df_feature.parquet')

# Test set

In [ ]:
gsmap_df_test = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/Test/GSMap_now_station_2021-2024.csv")
humid_df_test = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/Test/Humidity_2021-2024.csv")
pressure_df_test = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/Test/Pressure_2021-2024.csv")
temp_df_test = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/Test/Temperature_2021-2024.csv")
rain_df_test = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/CleanedSolto8th.csv")

coordinate_df = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week6/Coor_HII_495sta.csv")

In [ ]:
test_df = format_gs_map_dataset(
    df_humid=humid_df_test,
    df_pressure=pressure_df_test,
    df_temperature=temp_df_test,
    df_gsmap=gsmap_df_test,
    # df_rain=rain_df_test
)

# test_df = insert_location(test_df, coordinate_df)

# test_df = TimeSeriesDataFrame.from_data_frame(test_df,
#                                                id_column='Station', 
#                                                timestamp_column='timestamp',
#                                                # static_features_df=static_feature_df,
#                                                )

In [ ]:
test_df

In [ ]:
test_df.isnull().sum()

In [ ]:
test_df = test_df.dropna()
test_df = test_df.reset_index(drop=True)
test_df.isnull().sum()

In [ ]:
test_df

In [ ]:
df_train.to_parquet('/project/ai901504-ai0004/501641_Big/week6/train_df_normal.parquet')
test_df.to_parquet('/project/ai901504-ai0004/501641_Big/week6/eval_df_normal.parquet')
# test_df.to_parquet('/project/ai901504-ai0004/501641_Big/week6/test_df_normal.parquet')